In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
clusters_path = f'{projdir}/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
clusters.head()

In [ ]:
ips_subtypes_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_subtypes = pd.read_csv(ips_subtypes_path, sep='\t', index_col=0)
ips_subtypes.shape

In [ ]:
ips_subtypes.head()

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/raw/mct.h5ad')

In [ ]:
adata

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']
adata.obs['ips_cluster'] = ips_subtypes.reindex(adata.obs.index)['ips_subtype']

In [ ]:
mask = adata.obs['ips_cluster'].isna()
ips_adata = adata[~mask]
ips_adata

In [ ]:
ips_adata.obs['ips_cluster'].value_counts()

In [ ]:
tmp_adata = ips_adata.copy()

In [ ]:
sc.pp.normalize_total(tmp_adata)
sc.pp.log1p(tmp_adata)

In [ ]:
sc.tl.rank_genes_groups(tmp_adata, 'ips_cluster', method='wilcoxon', key_added = "wilcoxon")

In [ ]:
sc.pl.rank_genes_groups(tmp_adata, key='wilcoxon')

In [ ]:
main = sc.get.rank_genes_groups_df(tmp_adata, group='IPS_main', key='wilcoxon')
alt = sc.get.rank_genes_groups_df(tmp_adata, group='IPS_alt', key='wilcoxon')

In [ ]:
main["tmp_pvals_adj"] = main["pvals_adj"].replace(0, 1e-300)
main["neglog10_pvals"] = -np.log10(main["tmp_pvals_adj"])

alt["tmp_pvals_adj"] = alt["pvals_adj"].replace(0, 1e-300)
alt["neglog10_pvals"] = -np.log10(alt["tmp_pvals_adj"])

### volcano plot

In [ ]:
def volcano_plot(df, title="", lfc_thresh=0.5, pval_thresh=0.05):
    df = df.copy()

    # avoid log(0)
    df["pvals_adj"] = df["pvals_adj"].replace(0, 1e-300)

    df["neglog10_p"] = -np.log10(df["pvals_adj"])

    # classify points
    df["sig"] = "NS"
    df.loc[(df["logfoldchanges"] > lfc_thresh) & (df["pvals_adj"] < pval_thresh), "sig"] = "Up"
    df.loc[(df["logfoldchanges"] < -lfc_thresh) & (df["pvals_adj"] < pval_thresh), "sig"] = "Down"

    # plot
    plt.figure(figsize=(6,5))

    for label in ["NS", "Up", "Down"]:
        sub = df[df["sig"] == label]
        plt.scatter(sub["logfoldchanges"], sub["neglog10_p"], s=10, label=label)

    # thresholds
    plt.axvline(lfc_thresh, linestyle="--")
    plt.axvline(-lfc_thresh, linestyle="--")
    plt.axhline(-np.log10(pval_thresh), linestyle="--")

    plt.xlabel("log2 Fold Change")
    plt.ylabel("-log10(adj p-value)")
    plt.title(title)
    plt.legend()

    plt.show()

In [ ]:
def add_labels(df, n=10):
    top = df.nsmallest(n, "pvals_adj")
    for _, row in top.iterrows():
        plt.text(row["logfoldchanges"], row["neglog10_p"], row["names"], fontsize=8)

In [ ]:
volcano_plot(main, title="IPS_main vs rest")

In [ ]:
volcano_plot(alt, title="IPS_alt vs rest")

### write out

In [ ]:
main.to_csv(f'{projdir}/csv/degs/ips_subtypes/ips_main.csv', sep='\t')
alt.to_csv(f'{projdir}/csv/degs/ips_subtypes/ips_alt.csv', sep='\t')

In [ ]:
f'{projdir}/csv/degs/ips_subtypes/ips_alt.csv'

In [ ]:
!date